In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [26]:
customers=pd.read_csv(r"C:\Users\TUHIN\Downloads\Banking_Risk_Analytics_Dataset-Branches,Banking_Risk_Analytics_Dataset-Customers,Banking_Risk_A[...]\Banking_Risk_Analytics_Dataset-Customers.csv")
loans=pd.read_csv(r"C:\Users\TUHIN\Downloads\Banking_Risk_Analytics_Dataset-Branches,Banking_Risk_Analytics_Dataset-Customers,Banking_Risk_A[...]\Banking_Risk_Analytics_Dataset-Loans.csv")
transaction=pd.read_csv(r"C:\Users\TUHIN\Downloads\Banking_Risk_Analytics_Dataset-Branches,Banking_Risk_Analytics_Dataset-Customers,Banking_Risk_A[...]\Banking_Risk_Analytics_Dataset-Transactions.csv")



In [30]:
customers["Customer_Id"].duplicated().sum()

np.int64(0)

In [25]:
customer.dtypes

Customer_Id          object
Age                   int64
Gender               object
City                 object
Occupation           object
Education            object
Annual_income         int64
Credit_score          int64
Customer_segment     object
Branch_id            object
Unnamed: 10         float64
Unnamed: 11         float64
Unnamed: 12         float64
Unnamed: 13          object
Unnamed: 14         float64
dtype: object

In [31]:
loans["Customer_Id"].duplicated().sum()

np.int64(2898)

In [32]:
customers.isnull().sum()/len(customer)*100

Customer_Id           0.00
Age                   0.00
Gender                0.00
City                  0.00
Occupation            0.00
Education             0.00
Annual_income         0.00
Credit_score          0.00
Customer_segment      0.00
Branch_id             0.00
Unnamed: 10         100.00
Unnamed: 11         100.00
Unnamed: 12         100.00
Unnamed: 13          99.98
Unnamed: 14          99.98
dtype: float64

In [42]:
loans.isnull().sum()/len(loans)*100

loan_id                 0.0
Customer_Id             0.0
loan_type               0.0
application_date        0.0
loan_amount             0.0
interest_rate           0.0
tenure_months           0.0
loan_status             0.0
outstanding_amount      0.0
credit_score            0.0
annual_income           0.0
risk_band               0.0
debt_to_income_proxy    0.0
dtype: float64

In [35]:
transaction.isnull().sum()/len(transaction)*100

transaction_id        0.0
Customer_Id           0.0
transaction_date      0.0
transaction_type      0.0
amount                0.0
channel               0.0
transaction_status    0.0
dtype: float64

In [41]:
loans.dtypes

loan_id                  object
Customer_Id              object
loan_type                object
application_date         object
loan_amount               int64
interest_rate           float64
tenure_months             int64
loan_status              object
outstanding_amount      float64
credit_score              int64
annual_income             int64
risk_band                object
debt_to_income_proxy    float64
dtype: object

In [37]:
transaction.dtypes

transaction_id         object
Customer_Id            object
transaction_date       object
transaction_type       object
amount                float64
channel                object
transaction_status     object
dtype: object

In [40]:
customers = customers.loc[:, ~customers.columns.str.startswith("Unnamed")]
loans = loans.loc[:, ~loans.columns.str.startswith("Unnamed")]

In [45]:
customers["Customer_Id"].nunique()

5000

In [47]:
loans["Customer_Id"].nunique()

3602

In [48]:
transaction["Customer_Id"].nunique()

4988

In [50]:
loan_summary = loans.groupby("Customer_Id").agg(total_loans=("loan_id", "count"),
                total_loan_amount=("loan_amount", "sum"),
                total_outstanding=("outstanding_amount", "sum"),
                avg_interest_rate=("interest_rate", "mean"),
                avg_loan_amount=("loan_amount", "mean")
).reset_index()
loan_summary.sample(5)

,Customer_Id,total_loans,total_loan_amount,total_outstanding,avg_interest_rate,avg_loan_amount
327,C00460,2,1050000,761789.39,12.810,525000.0
361,C00508,1,2500000,1907233.93,15.840,2500000.0
1537,C02121,1,2500000,2150864.39,8.550,2500000.0
2792,C03905,1,2500000,745641.87,15.400,2500000.0
1472,C02036,2,500000,122843.36,12.985,250000.0


In [53]:
transaction_summary = transaction.groupby("Customer_Id").agg(total_transactions=("transaction_id", "count"),
                      total_transaction_amount=("amount", "sum"),
                      avg_transaction_amount=("amount", "mean")
).reset_index()
transaction_summary.sample(3)

,Customer_Id,total_transactions,total_transaction_amount,avg_transaction_amount
3384,C03392,6,29973.05,4995.508333
630,C00631,5,59597.03,11919.406000
2757,C02765,5,30920.39,6184.078000


In [59]:
successful_transactions = (transaction[transaction["transaction_status"] == "Success"].groupby("Customer_Id").size()
    .reset_index(name="successful_transactions"))

failed_transactions = (transaction[transaction["transaction_status"] != "Success"].groupby("Customer_Id")
    .size().reset_index(name="failed_transactions"))

successful_transactions.sample(5)

,Customer_Id,successful_transactions
3225,C03235,7
2838,C02848,1
1617,C01623,4
3664,C03674,1
2057,C02064,7


In [60]:
failed_transactions.sample(5)

,Customer_Id,failed_transactions
580,C02933,1
364,C01950,1
89,C00533,1
122,C00707,1
881,C04567,1


In [63]:
customer_loans = customers.merge(loan_summary,on="Customer_Id",how="left")
customer_loans.sample(3)

,Customer_Id,Age,Gender,City,Occupation,Education,Annual_income,Credit_score,Customer_segment,Branch_id,total_loans,total_loan_amount,total_outstanding,avg_interest_rate,avg_loan_amount
2354,C02355,55,Male,Kolkata,Salaried,Graduate,687000,622,Affluent,BR004,1.0,200000.0,0.0,14.8900,200000.0
2301,C02302,28,Male,Hyderabad,Professional,Post Graduate,299000,764,Mass Market,BR007,4.0,5750000.0,3251383.1,11.7675,1437500.0
2748,C02749,26,Male,Pune,Self Employed,Post Graduate,355000,744,Affluent,BR008,1.0,750000.0,0.0,8.4600,750000.0


In [64]:
df = customer_loans.merge(transaction_summary,on="Customer_Id",how="left")
df.sample(5)

,Customer_Id,Age,Gender,City,Occupation,Education,Annual_income,Credit_score,Customer_segment,Branch_id,total_loans,total_loan_amount,total_outstanding,avg_interest_rate,avg_loan_amount,total_transactions,total_transaction_amount,avg_transaction_amount
2923,C02924,63,Male,Mumbai,Salaried,Graduate,1079000,850,Affluent,BR008,3.0,900000.0,637919.45,9.826667,300000.0,6.0,33049.58,5508.263333
2234,C02235,28,Female,Hyderabad,Salaried,Graduate,354000,850,Affluent,BR006,2.0,2250000.0,1467883.19,13.800000,1125000.0,6.0,54506.82,9084.470000
1373,C01374,32,Male,Chennai,Salaried,Post Graduate,311000,719,Affluent,BR006,2.0,300000.0,150042.81,12.480000,150000.0,6.0,23836.19,3972.698333
1627,C01628,39,Male,Mumbai,Professional,Post Graduate,3004000,835,Affluent,BR001,1.0,100000.0,77037.19,11.470000,100000.0,7.0,69174.54,9882.077143
3292,C03293,47,Female,Bengaluru,Salaried,Under Graduate,342000,692,Affluent,BR007,NaN,NaN,NaN,NaN,NaN,7.0,28697.76,4099.680000


In [65]:
df.head()

,Customer_Id,Age,Gender,City,Occupation,Education,Annual_income,Credit_score,Customer_segment,Branch_id,total_loans,total_loan_amount,total_outstanding,avg_interest_rate,avg_loan_amount,total_transactions,total_transaction_amount,avg_transaction_amount
0,C00001,59,Male,Mumbai,Salaried,Graduate,1839000,722,Mass Market,BR002,3.0,2750000.0,380772.51,11.666667,916666.666667,4.0,7084.85,1771.2125
1,C00002,49,Male,Chennai,Salaried,Graduate,959000,784,Mass Market,BR005,2.0,2000000.0,731999.90,10.840000,1000000.000000,10.0,134676.19,13467.6190
2,C00003,35,Male,Delhi,Salaried,Post Graduate,396000,782,Mass Market,BR004,NaN,NaN,NaN,NaN,NaN,4.0,26174.46,6543.6150
3,C00004,63,Female,Kolkata,Business,Post Graduate,253000,658,Mass Market,BR002,1.0,500000.0,410368.40,16.270000,500000.000000,8.0,60246.30,7530.7875
4,C00005,28,Male,Bhubaneswar,Professional,Graduate,717000,615,Mass Market,BR001,NaN,NaN,NaN,NaN,NaN,5.0,56209.45,11241.8900


In [68]:
df.dtypes

Customer_Id                  object
Age                           int64
Gender                       object
City                         object
Occupation                   object
Education                    object
Annual_income                 int64
Credit_score                  int64
Customer_segment             object
Branch_id                    object
total_loans                 float64
total_loan_amount           float64
total_outstanding           float64
avg_interest_rate           float64
avg_loan_amount             float64
total_transactions          float64
total_transaction_amount    float64
avg_transaction_amount      float64
dtype: object

In [70]:
df.shape

(5000, 18)

In [71]:
df.nunique()

Customer_Id                 5000
Age                           45
Gender                         2
City                          10
Occupation                     4
Education                      3
Annual_income               1473
Credit_score                 284
Customer_segment               3
Branch_id                      8
total_loans                    6
total_loan_amount            116
total_outstanding           3035
avg_interest_rate           2062
avg_loan_amount              173
total_transactions            16
total_transaction_amount    4988
avg_transaction_amount      4987
dtype: int64

In [73]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_Id               5000 non-null   object 
 1   Age                       5000 non-null   int64  
 2   Gender                    5000 non-null   object 
 3   City                      5000 non-null   object 
 4   Occupation                5000 non-null   object 
 5   Education                 5000 non-null   object 
 6   Annual_income             5000 non-null   int64  
 7   Credit_score              5000 non-null   int64  
 8   Customer_segment          5000 non-null   object 
 9   Branch_id                 5000 non-null   object 
 10  total_loans               3602 non-null   float64
 11  total_loan_amount         3602 non-null   float64
 12  total_outstanding         3602 non-null   float64
 13  avg_interest_rate         3602 non-null   float64
 14  avg_loan

In [72]:
df.isnull().sum()/len(df)*100

Customer_Id                  0.00
Age                          0.00
Gender                       0.00
City                         0.00
Occupation                   0.00
Education                    0.00
Annual_income                0.00
Credit_score                 0.00
Customer_segment             0.00
Branch_id                    0.00
total_loans                 27.96
total_loan_amount           27.96
total_outstanding           27.96
avg_interest_rate           27.96
avg_loan_amount             27.96
total_transactions           0.24
total_transaction_amount     0.24
avg_transaction_amount       0.24
dtype: float64

In [74]:
df["total_loans"].describe()

count    3602.000000
mean        1.804553
std         0.963425
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         6.000000
Name: total_loans, dtype: float64

In [75]:
df["total_loans"].skew()

np.float64(1.235619924411998)